In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

database_name = "fsdh_artemis"

bronze_table_lines = f"{database_name}.bronze_oem_lines"

silver_table_header = f"{database_name}.silver_oem_header"
silver_table_metadata = f"{database_name}.silver_oem_metadata"
silver_table_state_vectors = f"{database_name}.silver_orion_state_vectors"

In [0]:
bronze_lines_df = spark.table(bronze_table_lines)

display(
    bronze_lines_df
    .orderBy("line_number")
)

In [0]:
classified_df = (
    bronze_lines_df
    .withColumn("raw_line_trimmed", F.trim(F.col("raw_line")))
    .withColumn("is_blank", F.col("raw_line_trimmed") == "")
    .withColumn("is_meta_start", F.col("raw_line_trimmed") == "META_START")
    .withColumn("is_meta_stop", F.col("raw_line_trimmed") == "META_STOP")
    .withColumn("is_comment", F.col("raw_line_trimmed").rlike(r"^COMMENT\b"))
    .withColumn("is_state_vector", F.col("raw_line_trimmed").rlike(r"^\d{4}-\d{2}-\d{2}T"))
    .withColumn(
        "is_key_value",
        F.col("raw_line_trimmed").rlike(r"^[A-Z0-9_]+\s*=")
    )
)
display(
    classified_df
    .select(
        "line_number",
        "raw_line",
        "raw_line_trimmed",
        "is_blank",
        "is_meta_start",
        "is_meta_stop",
        "is_comment",
        "is_key_value",
        "is_state_vector"
    )
    .orderBy("line_number")
)

In [0]:
meta_window = (
    Window
    .partitionBy("source_file")
    .orderBy("line_number")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

classified_with_meta_df = (
    classified_df
    .withColumn(
        "meta_start_count",
        F.sum(F.when(F.col("is_meta_start"), F.lit(1)).otherwise(F.lit(0))).over(meta_window)
    )
    .withColumn(
        "meta_stop_count",
        F.sum(F.when(F.col("is_meta_stop"), F.lit(1)).otherwise(F.lit(0))).over(meta_window)
    )
    .withColumn(
        "inside_meta_block",
        F.col("meta_start_count") > F.col("meta_stop_count")
    )
)

display(
    classified_with_meta_df
    .select(
        "line_number",
        "raw_line_trimmed",
        "is_meta_start",
        "is_meta_stop",
        "is_key_value",
        "inside_meta_block",
        "is_state_vector"
    )
    .orderBy("line_number")
)

In [0]:
header_df = (
    classified_with_meta_df
    .filter(F.col("is_key_value"))
    .filter(~F.col("inside_meta_block"))
    .withColumn("split_parts", F.split(F.col("raw_line_trimmed"), r"\s*=\s*", 2))
    .select(
        "source_file",
        "line_number",
        F.col("split_parts")[0].alias("header_key"),
        F.col("split_parts")[1].alias("header_value")
    )
)

display(header_df.orderBy("line_number"))

In [0]:
metadata_df = (
    classified_with_meta_df
    .filter(F.col("is_key_value"))
    .filter(F.col("inside_meta_block"))
    .withColumn("split_parts", F.split(F.col("raw_line_trimmed"), r"\s*=\s*", 2))
    .select(
        "source_file",
        "line_number",
        F.col("split_parts")[0].alias("meta_key"),
        F.col("split_parts")[1].alias("meta_value")
    )
)

display(metadata_df.orderBy("line_number"))

In [0]:
state_vectors_df = (
    classified_with_meta_df
    .filter(F.col("is_state_vector"))
    .withColumn("parts", F.split(F.col("raw_line_trimmed"), r"\s+"))
    .select(
        "source_file",
        "line_number",
        F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd'T'HH:mm:ss.SSS").alias("epoch_utc"),
        F.col("parts")[1].cast("double").alias("x_km"),
        F.col("parts")[2].cast("double").alias("y_km"),
        F.col("parts")[3].cast("double").alias("z_km"),
        F.col("parts")[4].cast("double").alias("vx_km_s"),
        F.col("parts")[5].cast("double").alias("vy_km_s"),
        F.col("parts")[6].cast("double").alias("vz_km_s")
    )
)

display(state_vectors_df.orderBy("line_number"))

In [0]:
comments_df = (
    classified_with_meta_df
    .filter(F.col("is_comment"))
    .select(
        "source_file",
        "line_number",
        F.regexp_replace(F.col("raw_line_trimmed"), r"^COMMENT\s*", "").alias("comment_text")
    )
)

display(comments_df.orderBy("line_number"))

In [0]:
header_df.write.mode("overwrite").saveAsTable(silver_table_header)
metadata_df.write.mode("overwrite").saveAsTable(silver_table_metadata)
state_vectors_df.write.mode("overwrite").saveAsTable(silver_table_state_vectors)

print(f"Saved table: {silver_table_header}")
print(f"Saved table: {silver_table_metadata}")
print(f"Saved table: {silver_table_state_vectors}")

In [0]:
print("Header:")
display(spark.table(silver_table_header).orderBy("line_number"))

print("Metadata:")
display(spark.table(silver_table_metadata).orderBy("line_number"))

print("State vectors:")
display(spark.table(silver_table_state_vectors).orderBy("line_number"))
